# Быстрый старт RAG Pipeline

Этот ноутбук демонстрирует полный цикл подготовки и запуска RAG-системы:
1. Установка зависимостей (GPU A100 с 25 GB VRAM используется автоматически, при отсутствии GPU всё работает на CPU).
2. Скачивание и квантование выбранной LLM.
3. Построение RAG-индекса на собственных данных.
4. Загрузка вопросов из разных форматов.
5. Инференс и вывод красивой таблицы результатов.
6. Сохранение ответов в нужных форматах.

Все функции снабжены русскими docstring'ами — можно навести курсор и увидеть описание параметров.

**Для Google Colab или удаленной системы:** Запустите секцию "0. Настройка для Colab" перед началом работы.


## 0. Настройка для Colab / Удаленной системы

**Для Google Colab или удаленной системы:** Запустите эту секцию перед началом работы.

**Для локального запуска:** Пропустите эту секцию.


In [ ]:
# Для Colab/удаленной системы раскомментируйте:
# !git clone https://github.com/phantom2059/baseline_rag.git
# !cd baseline_rag && pwd
# import sys
# sys.path.insert(0, '/content/baseline_rag')
# %cd baseline_rag


### 0.2. Установка зависимостей из requirements.txt


In [ ]:
import sys
# Для Colab раскомментируйте следующую строку:
# sys.path.insert(0, '/content/baseline_rag')

%pip install -q -U pip
%pip install -q -r requirements.txt

# Для Colab используем faiss-gpu (faiss-gpu-cu121 может не работать)
# Для локального запуска faiss уже установлен из requirements.txt
try:
    %pip install -q faiss-gpu
except Exception:
    pass  # faiss-cpu уже установлен из requirements.txt

print("✓ Зависимости установлены")


### 0.3. Проверка GPU


In [ ]:
import torch

print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠ GPU не обнаружен. Включите GPU в настройках Colab или используйте CPU.")


## 1. Установка зависимостей (для локального запуска)

**Для Colab:** Используйте секцию 0.2 выше.

**Для локального запуска:** Запустите эту ячейку один раз. Если какой-то пакет уже установлен, команда просто обновит его до актуальной версии.


In [ ]:
%pip install -q -U pip
%pip install -q -r requirements.txt

# Для локального запуска с GPU попробуйте faiss-gpu-cu121
try:
    %pip install -q -U faiss-gpu-cu121
except Exception:
    %pip install -q -U faiss-cpu

print("✓ Зависимости установлены")


## 2. Загрузка и квантование модели
Укажите нужное имя модели на HuggingFace или прямую ссылку. Функция автоматически скачает веса, применит выбранное квантование и создаст `config.json`.


In [ ]:
from model_downloader import download_model

MODEL_NAME = "Gaivoronsky/Mistral-7B-Saiga"
MODEL_DIR = "./model"
DOWNLOAD_MODEL = True  # поставьте False, если модель уже скачана

if DOWNLOAD_MODEL:
    download_model(
        model_name=MODEL_NAME,
        model_dir=MODEL_DIR,
        quantize=True,
        bits=8,
        config_path="config.json",
    )
else:
    print("[SKIP] Скачивание модели пропущено — используем существующие веса.")


## 3. Создание RAG-индекса
Если у вас есть собственные данные, включите флаг `BUILD_RAG` и укажите путь к файлу. Поддерживаются форматы CSV, XLSX, JSON и TXT. Логика нормализации совпадает с боевым сервисом.


In [ ]:
from rag_builder import create_rag_index

BUILD_RAG = False  # включите True, чтобы пересобрать индекс
DATA_PATH = "data/input.csv"  # или .xlsx/.json/.txt
RAG_DIR = "./rag"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L12-v2"

if BUILD_RAG:
    create_rag_index(
        data_path=DATA_PATH,
        output_dir=RAG_DIR,
        embedding_model=EMBEDDING_MODEL,
        chunk_size=700,
        chunk_overlap=120,
    )
else:
    print("[SKIP] Пересборка RAG отключена — используем готовые артефакты.")


## 4. Загрузка вопросов
Укажите файл с вопросами (JSON массив/объект, CSV, XLSX или TXT). Все форматы определяются автоматически.


In [ ]:
from utils import load_questions

INPUT_FILE = "input.json"
questions = load_questions(INPUT_FILE)
print(f"[DATA] Загружено {len(questions)} вопросов")
preview_count = min(5, len(questions))
print("[DATA] Примеры:")
for idx in range(preview_count):
    print(f"  {idx+1}. {questions[idx]}")


## 5. Инференс
Создаём экземпляр `FactualModel`, который автоматически загрузит модель и RAG-артефакты. Для ускорения используется GPU A100 (если доступно).


In [ ]:
from factual_model import FactualModel

model = FactualModel()
answers = model.generate_batch(questions, batch_size=8)
print(f"[INFER] Получено ответов: {len(answers)}")


## 6. Таблица результатов
Ниже выводится таблица в стиле `saiga_benchmark.ipynb` с первыми примерами ответов.


In [ ]:
from tabulate import tabulate

rows = []
for idx, (q, a) in enumerate(zip(questions, answers), start=1):
    rows.append({
        "#": idx,
        "chars": len(a),
        "question": q if len(q) <= 120 else q[:117] + "...",
        "answer": a,
    })
print(tabulate(rows[:10], headers="keys", tablefmt="github"))


## 7. Сохранение результатов
Укажите формат и путь. Можно сохранить сразу несколько вариантов (JSON массив, JSON пары, CSV).


In [ ]:
from utils import save_results

OUTPUT_CSV = "output.csv"

save_results(questions, answers, OUTPUT_CSV, fmt="csv")
print("[SAVE] Результаты записаны в указанные файлы.")
